# Week 1 - Applicant Dataset Cleaning
### Name: Syed Muhammed Ahmed

**Objective:** Practice data cleaning, wrangling, and basic exploration using Pandas.

**Note on the data:** The dataset used here (`applicants.csv`) is a mock dataset I generated myself (using Faker), deliberately built with the exact issues this task asks us to practice fixing — missing values, duplicate applicants, inconsistent domain/status text, text stored dates, and messy phone number formatting — following the column structure and messiness guide given in the task brief.

In [1]:
import pandas as pd

## Step 2: Load the Dataset

In [2]:
df = pd.read_csv("applicants.csv")
df.head()

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Joshua Blair,joshua.blair@example.com,033465787133,Data Science,IBA Karachi,"August 28, 2025",Rejected
1,Dr. Steven Martin,dr..steven.martin@example.com,03064-7468723,Data Science,FAST-NUCES,2025-11-21,SELECTED
2,Sherri Edwards,sherri.edwards@example.com,03767 2423884,webdev,LUMS,05-05-2026,Under Review
3,Kathy Rivas,kathy.rivas@example.com,03872-7743487,Web Development,COMSATS,22-03-2026,Selected
4,Daniel Floyd,daniel.floyd@example.com,03957 7738721,MOBILE APP DEV,UET Lahore,"November 16, 2025",Selected


## Step 3: First Look - Understand Before You Clean

In [3]:
df.shape

(81, 7)

In [4]:
df.head()

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Joshua Blair,joshua.blair@example.com,033465787133,Data Science,IBA Karachi,"August 28, 2025",Rejected
1,Dr. Steven Martin,dr..steven.martin@example.com,03064-7468723,Data Science,FAST-NUCES,2025-11-21,SELECTED
2,Sherri Edwards,sherri.edwards@example.com,03767 2423884,webdev,LUMS,05-05-2026,Under Review
3,Kathy Rivas,kathy.rivas@example.com,03872-7743487,Web Development,COMSATS,22-03-2026,Selected
4,Daniel Floyd,daniel.floyd@example.com,03957 7738721,MOBILE APP DEV,UET Lahore,"November 16, 2025",Selected


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Applicant Name    79 non-null     str  
 1   Email             75 non-null     str  
 2   Phone             81 non-null     str  
 3   Domain Applied    81 non-null     str  
 4   University        76 non-null     str  
 5   Application Date  81 non-null     str  
 6   Status            77 non-null     str  
dtypes: str(7)
memory usage: 4.6 KB


In [6]:
df.isnull().sum()

Applicant Name      2
Email               6
Phone               0
Domain Applied      0
University          5
Application Date    0
Status              4
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(3)

**Observations (before cleaning):**
- The dataset has **81 rows and 7 columns**.
- `Applicant Name` has **2 missing values**.
- `Email` has **6 missing values**.
- `University` has **5 missing values**.
- `Status` has **4 missing values**.
- There are **3 exact duplicate rows**.
- `Application Date` is currently stored as **text (object type)**, not as an actual date, and includes mixed formats plus one clearly broken value.
- `Domain Applied` and `Status` both contain many inconsistent spelling/casing variants of the same underlying categories (e.g. `"web dev"`, `"WEB DEV"`, `"Web Development "` are all the same track).
- `Phone` numbers are stored with inconsistent dash/space formatting.

This is the "before" snapshot I'll compare against the cleaned result at the end.

## Step 4: Handle Missing Values (Column by Column, With Justification)

There is no single correct rule for every column — the right choice depends on how critical the field is.

**Applicant Name** is critical identifying information; a row without a name isn't usable for outreach, so I drop these rows rather than trying to guess or fill in a name.

In [8]:
df = df.dropna(subset=["Applicant Name"])
df.shape

(79, 7)

**Email** is also fairly critical (used for follow-up), but instead of losing the row entirely I filled missing emails with a clear placeholder, `"not_provided@example.com"`, so the row is still usable for other analysis (domain applied, status, etc.) even though we can't contact that applicant directly.

In [9]:
df["Email"] = df["Email"].fillna("not_provided@example.com")

**University** is less critical than name/email, so instead of dropping the row I filled the gap with a placeholder, `"Not Specified"`.

In [10]:
df["University"] = df["University"].fillna("Not Specified")

**Status** missing likely means the application simply hasn't been processed yet, so I filled it with a sensible default, `"Under Review"`, rather than dropping the row.

In [11]:
df["Status"] = df["Status"].fillna("Under Review")

In [12]:
# Confirm no missing values remain in these columns
df.isnull().sum()

Applicant Name      0
Email               0
Phone               0
Domain Applied      0
University          0
Application Date    0
Status              0
dtype: int64

## Step 5: Remove Duplicate Applicants

In [13]:
# Check exact duplicates first
df.duplicated().sum()

np.int64(3)

In [14]:
df = df.drop_duplicates()
df.shape

(76, 7)

`drop_duplicates()` alone only catches rows that are 100% identical across every column, so it won't catch the same person applying twice with slightly different formatting (extra whitespace, a different phone format, etc.). To catch those near-duplicates, I check for repeated `Email` values instead, since email is a much more reliable identifier than `Applicant Name` (two different people can share a name, but not a real email address).

In [15]:
# Duplicate applications by the same email, before removing them
df.duplicated(subset=["Email"]).sum()

np.int64(7)

In [16]:
df = df.drop_duplicates(subset=["Email"], keep="first")
df.shape

(69, 7)

I removed the exact duplicate rows first, then removed additional near-duplicate applications identified by matching `Email`, keeping the first occurrence of each. I chose `Email` over full-row or name-based matching because it's the field most likely to uniquely identify a real applicant, while still catching cases where whitespace or formatting differed row to row.

## Step 6: Standardize Text Fields

In [17]:
# Strip extra whitespace and standardize casing on the applicant name (Title Case)
df["Applicant Name"] = df["Applicant Name"].str.strip().str.title()

# Standardize email casing (lowercase, since emails are case-insensitive in practice)
df["Email"] = df["Email"].str.strip().str.lower()

In [18]:
df["Domain Applied"].unique()

<StringArray>
[          'Data Science',                 'webdev',        'Web Development',
         'MOBILE APP DEV',              'UX Design', 'Mobile App Development',
          ' Data Science',         'cyber security',           'UI/UX Design',
         'Cyber Security',          'Cybersecurity',         'mobile app dev',
           'ui/ux design', 'Mobile app development',           'UI/UX DESIGN',
                'web dev',       'Web Development ',           'data science',
                'WEB DEV',          'CYBERSECURITY',           'DATA SCIENCE']
Length: 21, dtype: str

`Domain Applied` has many spelling/casing variants of the same 5 real tracks. I strip whitespace and lowercase everything first, then map every variant to one final, clean label.

In [19]:
df["Domain Applied"] = df["Domain Applied"].str.strip().str.lower()

domain_mapping = {
    "web dev": "Web Development",
    "webdev": "Web Development",
    "web development": "Web Development",
    "data science": "Data Science",
    "mobile app dev": "Mobile App Development",
    "mobile app development": "Mobile App Development",
    "ui/ux design": "UI/UX Design",
    "ux design": "UI/UX Design",
    "cyber security": "Cybersecurity",
    "cybersecurity": "Cybersecurity",
}

df["Domain Applied"] = df["Domain Applied"].map(domain_mapping)
df["Domain Applied"].unique()

<StringArray>
[          'Data Science',        'Web Development', 'Mobile App Development',
           'UI/UX Design',          'Cybersecurity']
Length: 5, dtype: str

In [20]:
# Confirm no NaNs slipped through (would mean an unmapped variant was missed)
df["Domain Applied"].isnull().sum()

np.int64(0)

After mapping, `Domain Applied` contains exactly the 5 correct values, and there are no new missing values, confirming every messy variant was caught by the mapping dictionary.

Now I standardize `Status` the same way, so `"selected"`, `"Selected"`, and `"SELECTED"` all become one consistent value.

In [21]:
df["Status"] = df["Status"].str.strip().str.title()
df["Status"].unique()

<StringArray>
['Rejected', 'Selected', 'Under Review']
Length: 3, dtype: str

## Step 7: Fix Data Types (Especially Dates and Phone Numbers)

In [22]:
df["Application Date"] = pd.to_datetime(
    df["Application Date"],
    errors="coerce",
    format="mixed",
    dayfirst=True,
)
df["Application Date"].isnull().sum()

np.int64(1)

I used `errors="coerce"` so that any unreadable date format becomes `NaT` (Pandas' missing-date marker) instead of crashing the whole notebook. This column mixes several different date formats (`"2025-11-01"`, `"15-04-2026"`, `"October 27, 2025"`, etc.), so I also passed `format="mixed"` — without it, pandas can't reliably infer a single format across rows and will silently turn far more rows into `NaT` than actually have bad data. `dayfirst=True` tells pandas to read ambiguous numeric dates like `"03-01-2026"` as day-month-year rather than month-day-year. Checking `isnull().sum()` afterward tells us exactly how many original values genuinely couldn't be parsed. In this dataset, only one row had a clearly broken date value (`"not_a_date"`), which became `NaT` — I chose to keep that row (the rest of its data is still useful) rather than drop it, since a single missing date doesn't make the whole application unusable.

In [23]:
# Clean up Phone numbers - remove dashes and spaces, keep as text since phone numbers
# with leading zeros shouldn't be converted to actual numbers
df["Phone"] = df["Phone"].str.replace("-", "", regex=False).str.replace(" ", "", regex=False)
df["Phone"].head()

0    033465787133
1    030647468723
2    037672423884
3    038727743487
4    039577738721
Name: Phone, dtype: str

In [24]:
df.info()

<class 'pandas.DataFrame'>
Index: 69 entries, 0 to 80
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    69 non-null     str           
 1   Email             69 non-null     str           
 2   Phone             69 non-null     str           
 3   Domain Applied    69 non-null     str           
 4   University        69 non-null     str           
 5   Application Date  68 non-null     datetime64[us]
 6   Status            69 non-null     str           
dtypes: datetime64[us](1), str(6)
memory usage: 4.3 KB


`Application Date` is now confirmed as `datetime64` type instead of plain text (object), and `Phone` numbers are consistently formatted with no dashes or spaces.

## Step 8: Data Quality Summary

**Starting dataset:** 81 rows, 7 columns

**Issues found:**
- 2 missing values in Applicant Name, 6 in Email, 5 in University, 4 in Status
- 3 exact duplicate rows, plus 7 additional duplicate applications identified by matching Email
- `Domain Applied` had 21 different spelling/casing variations for only 5 real domains
- `Status` had multiple casing variants (e.g. "selected", "SELECTED", "Selected") for 3 real statuses
- `Application Date` was stored as text, not as an actual date type, mixed several different formats (ISO, DD-MM-YYYY, DD/MM/YYYY, "Month DD, YYYY"), and included one clearly unparseable value
- `Phone` numbers contained inconsistent dash/space formatting

**Actions taken:**
- Dropped 2 rows with missing Applicant Name (unusable without identification)
- Filled missing Email with "not_provided@example.com" (kept the row, flagged as unreachable)
- Filled missing University with "Not Specified" (non-critical field)
- Filled missing Status with "Under Review" (default for unprocessed applications)
- Removed 3 exact duplicate rows, then removed 7 further duplicates by matching Email (kept first)
- Standardized all Domain Applied values into 5 consistent categories
- Standardized Status into 3 consistent Title Case values
- Converted Application Date to proper datetime format using `format="mixed", dayfirst=True` (the 1 unparseable value became NaT and was kept)
- Cleaned Phone number formatting for consistency (no dashes/spaces)

**Final dataset:** 69 rows, 7 columns - no missing values in critical fields, no duplicate applicants, consistent text formatting throughout, and correct data types (`Application Date` is `datetime64`).

In [25]:
df.shape

(69, 7)

## Step 9: Export the Cleaned Dataset

In [26]:
df.to_csv("applicants_cleaned.csv", index=False)
print("Saved applicants_cleaned.csv with shape:", df.shape)

Saved applicants_cleaned.csv with shape: (69, 7)


## Step 10: Push to GitHub

Upload the following two files to a public repository:
- ✅ `Week1_Data_Cleaning_Ahmed.ipynb`
- ✅ `applicants_cleaned.csv`